# QLoRA fine-tune v2: Qwen3-8B for Text-to-SQL (benchmark v2, prompt v2)

Runs **unattended** on Kaggle. Hit `Save Version` -> `Save & Run All (Commit)`,
close the tab, come back to a finished adapter.

## Before running, set these in the right-hand panel

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (needed to download the model) |
| Input dataset | your uploaded `text2sql-sft-v2` dataset |

Only one T4 is used. The x2 option is selected because it is the
faster accelerator available on the free tier — the P100 has no tensor
cores and is slower for fp16 work despite its reputation.

| | |
|---|---|
| Base model | `Qwen/Qwen3-8B` @ `b968826d9c46` |
| Method | QLoRA — 4-bit NF4 base, LoRA r=16 |
| Training data | 2,175 examples (benchmark v2), frozen and hash-verified |
| Expected | ~5.5-6.5 h for 1 epoch (136 steps at ~140-160 s), inside Kaggle's 12 h limit |


## What is different from v1

Same base model, same QLoRA settings, same script. Only the data changed:
the v2 benchmark makes the column, money and month conventions consistent,
the v2 prompt states them (plus a reference date and a business glossary),
and five train-only templates add `PARTITION BY` and day-arithmetic, which
no v1 training example contained. Every v1 test template keeps its split.

## Why the first v2 attempt failed, and what changed

Version 1 ran out of GPU memory at step 34/136: 13.4 GB in use on a 14.6 GB T4,
with training otherwise healthy (loss 4.13 -> 0.047). v2 prompts are ~330 tokens
longer than v1 (max 1,944 vs 1,602) and the headroom was gone.

`train_qlora.py` now computes the output logits only at the ~45 supervised SQL
positions instead of all ~1,900 (`logits_to_keep`), which removes ~3 GB of
151k-vocabulary logits and their fp32 copies from every step. The loss is
identical; only wasted work is gone. It also pins one GPU (DataParallel would
scatter the position index) and accumulates 16 micro-batches, matching v1's
effective batch.

Version 2 got past the memory limit (step 50, 5 GB free) and failed at the
first evaluation: the Liger kernel's forward skips logits in eval mode when no
labels are passed. The loss now asks for them explicitly (`skip_logits=False`).

**If a run fails, its checkpoints are now kept.** Add the failed version as an
Input of the next run and cell 4 resumes from the latest `checkpoint-*`.


## 1. Environment check

Fails loudly and immediately if the GPU or the dataset is missing, rather
than after a 16 GB model download.


In [ ]:
import subprocess, os, sys

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU T4 x2'
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1024**3:.1f} GB | compute {p.major}.{p.minor}')

print()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(' ', os.path.join(root, f))

assert os.path.isdir('/kaggle/input') and any(os.scandir('/kaggle/input')), \
    'No input dataset attached. Add your text2sql-sft dataset in the right panel.'


## 2. Install dependencies

Loose lower bounds on purpose. Pinning exact versions is what broke this
on Colab: the base image ships a newer CUDA/Triton than old `bitsandbytes`
builds support. The training script records the resolved versions instead.


In [ ]:
# liger-kernel provides a fused cross-entropy that never materialises
# the full 151k-vocab logits tensor - without it the loss step OOMs
# on a 16 GB T4.
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl sentencepiece liger-kernel 2>&1 | tail -3

import importlib.metadata as md
for pkg in ['torch','transformers','peft','bitsandbytes','accelerate','liger-kernel']:
    try:    print(f'{pkg:<16}{md.version(pkg)}')
    except Exception: print(f'{pkg:<16}MISSING')


## 3. Verify the dataset against the frozen hashes

These are the content hashes recorded when the SFT dataset was built. If
they do not match, the training data is not the audited data and any
comparison against the frozen 10.82 % baseline would be meaningless — so
this aborts rather than warns.


In [ ]:
import json, hashlib, os

# Content hashes written by scripts/prepare_sft_dataset.py --version v2
EXPECTED = {'train': '5003da4e7e0d1825', 'validation': '91195bc38f324533'}

def find(name):
    for root, _, files in os.walk('/kaggle/input'):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

PATHS = {s: find(f'{s}.jsonl') for s in EXPECTED}
SCRIPT = find('train_qlora.py')

for split, expected in EXPECTED.items():
    rows = [json.loads(l) for l in open(PATHS[split], encoding='utf-8')]
    h = hashlib.sha256()
    for r in rows:
        for m in r['messages']:
            h.update(m['role'].encode()); h.update(bytes([0]))
            h.update(m['content'].encode()); h.update(bytes([1]))
    got = h.hexdigest()[:16]
    status = 'MATCH' if got == expected else 'MISMATCH'
    print(f'{split:<11}{len(rows):>5} records  {got}  {status}')
    assert got == expected, f'{split} differs from the frozen dataset'

print()
print('script :', SCRIPT)
print('verified against the frozen v2 SFT dataset')


## 4. Train

One epoch, effective batch 8, checkpoints every 50 steps into
`/kaggle/working/`. Measured throughput on a T4 was ~75 s per optimiser
step at batch 1, so expect roughly 5.5 hours — comfortably inside the 12 h
commit limit.

Config is the one proven working on real hardware. `--batch-size 2` was
tried and OOMed once gradient checkpointing was disabled; checkpointing
stays on.


In [ ]:
import os, subprocess, sys, glob, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # newer name

# Single GPU (see train_qlora.py) with accumulation 16 -> effective batch 16,
# the same as the v1 run, so the two adapters differ only in their data.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
OUT = '/kaggle/working/qwen3-8b-text2sql-qlora-v2'

# Resume: if a previous version of this notebook is attached as an input and
# it kept checkpoints, continue from the latest one instead of starting over.
resume = []
found = sorted(glob.glob('/kaggle/input/**/checkpoint-*', recursive=True),
               key=lambda p: int(p.rsplit('-', 1)[1]))
if found:
    latest = found[-1]
    os.makedirs(OUT, exist_ok=True)
    dst = os.path.join(OUT, os.path.basename(latest))
    if not os.path.exists(dst):
        shutil.copytree(latest, dst)
    resume = ['--resume']
    print('resuming from', latest, flush=True)

cmd = [sys.executable, SCRIPT,
       '--train-file', PATHS['train'],
       '--val-file', PATHS['validation'],
       '--output-dir', OUT,
       '--epochs', '1', '--grad-accum', '16'] + resume
print(' '.join(cmd), flush=True)

# The training process is run as a subprocess and its exit code is checked -
# a `!python` line that crashes would let the notebook carry on and Kaggle
# would mark the commit Successful.
proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1)
print(proc.stdout)

adapter = os.path.join(OUT, 'final_adapter')
TRAIN_OK = proc.returncode == 0 and os.path.isdir(adapter)

# A failed commit loses /kaggle/working entirely - two v2 attempts each lost
# 2 h of checkpoints that way. So a failure here is reported loudly and
# recorded in a marker file, but the notebook is allowed to finish, which is
# what makes the checkpoints survive for a --resume on the next version.
# Nothing downstream can mistake it for success: the packaging cell produces
# `checkpoints.zip` instead of `adapter.zip`, and the generation notebook
# refuses to run without final_adapter/adapter_config.json.
if not TRAIN_OK:
    tail = proc.stdout[-4000:]
    os.makedirs(OUT, exist_ok=True)
    with open(os.path.join(OUT, 'TRAINING_FAILED.txt'), 'w') as fh:
        fh.write(f'exit code {proc.returncode}' + chr(10) + tail)
    print()
    print('!' * 68)
    print(f'!! TRAINING FAILED (exit code {proc.returncode}) - see the log above')
    print('!! checkpoints (if any) are kept in the Output tab for --resume')
    print('!' * 68)
else:
    print()
    print('training finished, adapter written')


## 5. Results

Everything under `/kaggle/working/` is saved with the notebook version and
can be downloaded from the Output tab afterwards.


In [ ]:
import json, os
OUT = '/kaggle/working/qwen3-8b-text2sql-qlora-v2'

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p)/1e6:8.1f} MB  {p}')

try:
    m = json.load(open(f'{OUT}/training_metrics.json'))
    print()
    print('runtime         :', m.get('train_runtime_h'), 'h')
    print('sec/optim step  :', m.get('sec_per_optimizer_step'))
    print('final train loss:', m.get('train_loss'))
    print('final eval      :', m.get('final_eval'))
except FileNotFoundError:
    print('metrics missing - training did not finish')


## 6. Shrink the output for download

Intermediate checkpoints are large and redundant once the final adapter
exists. This keeps the adapter plus the config and metrics, which is all
Phase 10 needs.


In [ ]:
import shutil, os, glob
OUT = '/kaggle/working/qwen3-8b-text2sql-qlora-v2'
adapter = os.path.join(OUT, 'final_adapter')

if os.path.isdir(adapter):
    for ck in glob.glob(f'{OUT}/checkpoint-*'):
        shutil.rmtree(ck, ignore_errors=True)
        print('removed', os.path.basename(ck))
    shutil.make_archive('/kaggle/working/adapter', 'zip', OUT)
    print()
    print('download /kaggle/working/adapter.zip from the Output tab')
    print(f"size: {os.path.getsize('/kaggle/working/adapter.zip')/1e6:.1f} MB")
else:
    # Failed run: keep the checkpoints so the next version can resume.
    # Deliberately a different filename - this is NOT an adapter.
    cks = sorted(glob.glob(f'{OUT}/checkpoint-*'))
    print('no final_adapter - training did not finish')
    print('checkpoints kept for resume:', [os.path.basename(c) for c in cks] or 'none')
    print('to resume: add THIS notebook version as an Input of the next run')


---

## If the session is cut short

Checkpoints persist in the notebook output. Re-run with `--resume` added to
cell 4 and it continues from the last checkpoint instead of restarting.

## What this does not do

- No benchmark run — the 453-example test set is evaluated in Phase 10,
  through the same harness that produced the frozen baseline.
- No dataset modification — inputs are read-only on Kaggle by design.
- No CPU fallback — missing CUDA exits immediately.

## The detail that matters

Each example is ~1,473 prompt tokens and ~45 completion tokens, and the
prompt is ~97 % schema identical across all 2,133 examples. The script
masks the prompt out of the loss and trains on the SQL only. Without that,
almost all of the gradient would go into memorising a schema the model is
handed at inference — and the loss curve would look excellent while the
ability you care about barely moved. Startup prints the supervised token
share; expect ~3 %.
